In [0]:
%pip install FlagEmbedding

In [0]:
# ============================================================================
# TOPIC MODELING NOTEBOOK — Cell 1: Setup + load from deviation_embeddings
#
# SOURCE = deviation_embeddings  (NOT deviation_embed_input) because it is the
# only table holding BOTH the precomputed BGE-M3 vectors AND the text:
#   • BERTopic  → reuses the `embedding` (fine) column  → no re-encoding
#   • NMF       → uses the `embedding_text` column      → bag-of-words
# ============================================================================
import numpy as np
from pyspark.sql import functions as F

CATALOG = "us_gmsgq_dev"          # ← match your env
ALYT    = "gms_us_alyt"
EMB_TABLE = f"{CATALOG}.{ALYT}.deviation_embeddings"

pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "embedding_text", "core_embedding_text", "embedding")
    .toPandas()
)
pdf["embedding_text"] = pdf["embedding_text"].fillna("")

import re

def _strip_for_tfidf(text):
    """Remove structural scaffolding from embedding_text so c-TF-IDF sees only content.
    The embeddings (precomputed) already encode this context — we only need clean
    text for the CountVectorizer keyword-labeling step."""
    # Remove everything after the injected-reference section
    text = re.split(r"=== RESOLVED REFERENCES ===", text, maxsplit=1)[0]
    # Remove column-name prefixes ("Event_Title: ", "Root_Cause_SubCategory: ", etc.)
    text = re.sub(
        r"\b(Event_Title|Event_Description|Impact_Assessment|Quality_Final_Assessment"
        r"|Root_Cause_Category|Root_Cause_SubCategory|Action_Text)\s*:",
        "", text
    )
    return text.strip()

# docs  = text for c-TF-IDF labeling only (cleaned of structural noise)
# embs  = precomputed fine-tier BGE-M3 vectors (drive UMAP + HDBSCAN clustering)
docs = [_strip_for_tfidf(t) for t in pdf["embedding_text"].tolist()]
embs = np.array(pdf["embedding"].tolist(), dtype=np.float32)

print(f"Docs: {len(docs):,}  |  Embedding matrix: {embs.shape}")
assert embs.shape[1] == 1024, "expected 1024-dim BGE-M3 vectors"

In [0]:
%pip install bertopic

In [0]:
%pip install umap-learn

In [0]:
# ============================================================================
# Cell 2 — BERTopic on precomputed BGE-M3 embeddings
# Params tuned for a SMALL corpus (~1.1k rows): otherwise HDBSCAN dumps
# most docs into the -1 (outlier) bucket.
# ============================================================================
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# UMAP: cosine because BGE-M3 dense vectors are normalized; small n_neighbors
# for a small corpus so local structure isn't washed out.
umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0,
    metric="cosine", random_state=42,
)

# HDBSCAN: small min_cluster_size so you actually get topics on ~1.1k rows.
hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5,
    metric="euclidean", cluster_selection_method="eom",
    prediction_data=True,
)

# c-TF-IDF vectorizer: standard English stopwords PLUS structural terms from the
# embedding_text format (column names, reference markers, entity-type labels).
# NOTE: this is the LABELING step (bag-of-words), NOT the embedding step.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number", "root cause", "actiontext", "eventtitle", "impactassessment", "qualityfinalassessment", "rootcausecategory", "rootcausesubcategory", "eventdescription"
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS
vectorizer_model = CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2))

# BERTopic is used below to fit and transform the docs using precomputed embeddings.
topic_model = BERTopic(
    embedding_model=None,                 # ← precomputed; do NOT re-embed
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs, embeddings=embs)

info = topic_model.get_topic_info()
print(f"Topics found (excl. -1): {len(info[info.Topic != -1])}")
print(f"Outliers (-1): {int((np.array(topics) == -1).sum())} "
      f"({100*(np.array(topics)==-1).mean():.1f}%)")
display(info.head(30))

In [0]:
# ============================================================================
# Hyperparameter Tuning — UMAP (n_neighbors, n_components, min_dist)
# Grid search evaluating topic count, outlier %, confidence, and diversity.
# HDBSCAN params held fixed; only UMAP geometry is varied.
# ============================================================================
import itertools, time
import pandas as pd
import plotly.express as px
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# --- Search grid ---
param_grid = {
    "n_neighbors": [5, 10, 15, 25],
    "n_components": [3, 5, 10, 15],
    "min_dist":     [0.0, 0.05, 0.1, 0.25],
}

# Fixed HDBSCAN + vectorizer (only tuning UMAP here)
hdbscan_fixed = HDBSCAN(
    min_cluster_size=10, min_samples=5,
    metric="euclidean", cluster_selection_method="eom",
    prediction_data=True,
)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number",
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS
vectorizer_fixed = CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2))

results = []
combos = list(itertools.product(
    param_grid["n_neighbors"],
    param_grid["n_components"],
    param_grid["min_dist"],
))
print(f"Running {len(combos)} UMAP parameter combinations on {len(docs):,} docs...")

for i, (nn, nc, md) in enumerate(combos):
    t0 = time.time()
    umap_m = UMAP(n_neighbors=nn, n_components=nc, min_dist=md,
                  metric="cosine", random_state=42)

    model = BERTopic(
        embedding_model=None,
        umap_model=umap_m,
        hdbscan_model=hdbscan_fixed,
        vectorizer_model=vectorizer_fixed,
        calculate_probabilities=True,
        verbose=False,
    )
    topics_i, probs_i = model.fit_transform(docs, embeddings=embs)
    elapsed = time.time() - t0

    n_topics = len(set(topics_i)) - (1 if -1 in topics_i else 0)
    outlier_pct = 100 * (np.array(topics_i) == -1).mean()
    avg_prob = probs_i.max(axis=1).mean() if probs_i is not None else 0.0

    # Topic diversity: proportion of unique words in top-5 keywords across all topics
    all_words = []
    for t in set(topics_i):
        if t == -1: continue
        all_words.extend([w for w, _ in model.get_topic(t)[:5]])
    diversity = len(set(all_words)) / max(len(all_words), 1)

    results.append({
        "n_neighbors": nn, "n_components": nc, "min_dist": md,
        "n_topics": n_topics, "outlier_pct": round(outlier_pct, 1),
        "avg_prob": round(avg_prob, 4), "diversity": round(diversity, 4),
        "time_s": round(elapsed, 1),
    })

    if (i + 1) % 16 == 0:
        print(f"  [{i+1}/{len(combos)}] done...")

results_df = pd.DataFrame(results).sort_values(
    ["n_topics", "outlier_pct", "diversity"],
    ascending=[False, True, False],
).reset_index(drop=True)

print(f"\n\u2713 Grid search complete. Top configurations:")
display(results_df.head(20))

# --- Parallel coordinates plot to visualise trade-offs ---
fig = px.parallel_coordinates(
    results_df,
    dimensions=["n_neighbors", "n_components", "min_dist",
                "n_topics", "outlier_pct", "diversity", "avg_prob"],
    color="n_topics",
    color_continuous_scale=px.colors.sequential.Viridis,
    title="UMAP Hyperparameter Sweep — Parallel Coordinates",
    height=500,
)
fig.show()

# --- Heatmap: n_neighbors vs n_components (averaged over min_dist) ---
heat = results_df.groupby(["n_neighbors", "n_components"]).agg(
    topics_mean=("n_topics", "mean"),
    outlier_mean=("outlier_pct", "mean"),
).reset_index()

fig2 = px.density_heatmap(
    heat, x="n_neighbors", y="n_components", z="topics_mean",
    histfunc="avg", color_continuous_scale="YlOrRd",
    title="Avg Topics Found: n_neighbors × n_components (avg over min_dist)",
    height=400,
)
fig2.show()

In [0]:
# ============================================================================
# Re-fit with best hyperparams: n_neighbors=5, n_components=5, min_dist=0.0
# (34 topics, 15.2% outlier, 0.5063 avg_prob, 0.8118 diversity)
# Then 3D UMAP scatter with hover showing event description.
# ============================================================================
import plotly.express as px
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# --- Best UMAP params from grid search ---
BEST_NN = 5
BEST_NC = 5
BEST_MD = 0.0

umap_best = UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                 metric="cosine", random_state=42)

hdbscan_model = HDBSCAN(
    min_cluster_size=10, min_samples=5,
    metric="euclidean", cluster_selection_method="eom",
    prediction_data=True,
)

STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number",
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS
vectorizer_model = CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_best,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings=embs)

info = topic_model.get_topic_info()
n_real_topics = len(info[info.Topic != -1])
n_outliers = int((np.array(topics) == -1).sum())
print(f"Topics found: {n_real_topics}  |  "
      f"Outliers: {n_outliers} ({100*n_outliers/len(topics):.1f}%)")

# ============================================================================
# TOPIC SUMMARY TABLE — suggested names from top keywords
# ============================================================================
topic_summary = []
for _, row in info.iterrows():
    t = row["Topic"]
    if t == -1:
        topic_summary.append({
            "Topic": -1,
            "Suggested Name": "(Outlier / Unclassified)",
            "Count": row["Count"],
            "Top Keywords": "",
        })
    else:
        kws = [w for w, _ in topic_model.get_topic(t)[:8]]
        # Suggested name: capitalise the top 2-3 keywords as a short label
        suggested = " / ".join(kws[:3]).title()
        topic_summary.append({
            "Topic": t,
            "Suggested Name": suggested,
            "Count": row["Count"],
            "Top Keywords": ", ".join(kws),
        })

summary_df = pd.DataFrame(topic_summary)
print("\n" + "="*80)
print("TOPIC CLUSTERS  (suggested names derived from c-TF-IDF top keywords)")
print("="*80)
display(summary_df)

# ============================================================================
# 3D UMAP SCATTER — coloured by topic name
# NOTE: UMAP axes are NOT interpretable. They only encode neighbourhood
# proximity: points near each other have similar embeddings. The axis
# values themselves have no semantic meaning.
# ============================================================================
umap_3d = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
               metric="cosine", random_state=42)
coords_3d = umap_3d.fit_transform(embs)

# Build short topic labels for the scatter legend
topic_short_labels = {}
for _, row in summary_df.iterrows():
    t = int(row["Topic"])
    if t == -1:
        topic_short_labels[t] = "Outlier"
    else:
        topic_short_labels[t] = f"{t}: {row['Suggested Name']}"

plot_df = pd.DataFrame({
    "x": coords_3d[:, 0],
    "y": coords_3d[:, 1],
    "z": coords_3d[:, 2],
    "topic_name": [topic_short_labels[t] for t in topics],
    "pr_id": pdf["pr_id"].values,
    "event_description": [d[:300] + ("..." if len(d) > 300 else "") for d in docs],
    "keywords": [summary_df.loc[summary_df["Topic"]==t, "Top Keywords"].iloc[0] if t in summary_df["Topic"].values else "" for t in topics],
})

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="topic_name",
    hover_data={
        "pr_id": True,
        "event_description": True,
        "keywords": True,
        "x": False, "y": False, "z": False,
    },
    title=f"Topic Clusters ({n_real_topics} topics) — 3D UMAP projection (axes = proximity only, not interpretable)",
    opacity=0.75,
    height=750,
)
fig.update_traces(marker_size=3.5)
fig.update_layout(
    legend_title_text="Topic",
    hoverlabel=dict(font_size=10),
    scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""),  # blank axes since they're meaningless
)
fig.show()

In [0]:
# ============================================================================
# Guided BERTopic — Root_Cause_Category values as seed topics
#
# Semi-supervised approach: use the organisation's existing deviation categories
# as priors. BERTopic will nudge topic representations toward these seeds while
# still allowing emergent clusters to form beyond the seed list.
#
# Seeded topics 0…N map 1:1 to seed_topic_list entries (in order).
# Topics > N are emergent (discovered without a seed).
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
from pyspark.sql import functions as F
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# ---- Pull Root_Cause_Category + SubCategory from source to build seeds ----
CATALOG = "us_gmsgq_dev"
MART    = "gms_us_mart"
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

cat_df = (
    spark.table(SOURCE_TABLE)
    .select("Root_Cause_Category", "Root_Cause_SubCategory")
    .filter("Root_Cause_Category IS NOT NULL AND TRIM(Root_Cause_Category) != ''")
    .groupBy("Root_Cause_Category")
    .agg(
        F.collect_set("Root_Cause_SubCategory").alias("subcategories"),
        F.count("*").alias("n"),
    )
    .orderBy(F.col("n").desc())
    .toPandas()
)

print(f"Found {len(cat_df)} Root_Cause_Category values in source data:")
for _, row in cat_df.head(20).iterrows():
    subs = [s for s in (row['subcategories'] if row['subcategories'] is not None else []) if s and str(s).strip()][:4]
    print(f"  {row['Root_Cause_Category']:<40} (n={row['n']:>5})  subs: {subs}")

# ---- Build seed keyword lists ----
# Each seed = lowercased meaningful words from the category name + top sub-category names
IGNORE_WORDS = {
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text", "other",
    "not", "applicable", "the", "and", "for", "was", "were", "that", "this",
    "with", "from", "but", "are", "has", "have", "been", "will",
}

def _extract_seeds(category, subcategories):
    """Extract seed keywords from category + subcategory names."""
    words = set()
    for w in str(category).lower().replace("/", " ").replace("-", " ").split():
        if len(w) >= 3 and w not in IGNORE_WORDS:
            words.add(w)
    for sub in (subcategories if subcategories is not None and len(subcategories) > 0 else [])[:6]:
        if sub and str(sub).strip():
            for w in str(sub).lower().replace("/", " ").replace("-", " ").split():
                if len(w) >= 4 and w not in IGNORE_WORDS:
                    words.add(w)
    return list(words)

seed_topic_list = []
seed_labels = []
for _, row in cat_df.iterrows():
    seeds = _extract_seeds(row["Root_Cause_Category"], row["subcategories"])
    if len(seeds) >= 2:
        seed_topic_list.append(seeds)
        seed_labels.append(row["Root_Cause_Category"])

print(f"\n{len(seed_topic_list)} seed topics constructed:")
for label, seeds in zip(seed_labels, seed_topic_list):
    print(f"  {label:<40} seeds: {seeds[:8]}")

# ---- Fit guided BERTopic ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

custom_stops = list(ENGLISH_STOP_WORDS) + list(IGNORE_WORDS)

guided_model = BERTopic(
    embedding_model=None,
    umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    seed_topic_list=seed_topic_list,       # ← GUIDED: seeds from Root_Cause_Category
    calculate_probabilities=True,
    verbose=True,
)

guided_topics, guided_probs = guided_model.fit_transform(docs, embeddings=embs)

guided_info = guided_model.get_topic_info()
n_guided = len(guided_info[guided_info.Topic != -1])
n_out = int((np.array(guided_topics) == -1).sum())
print(f"\nGuided topics: {n_guided}  |  Outliers: {n_out} ({100*n_out/len(guided_topics):.1f}%)")
print(f"  Seeded topics: {min(n_guided, len(seed_labels))}  |  Emergent: {max(0, n_guided - len(seed_labels))}")

# ---- Summary table: Topic → Seed Category → Discovered Keywords ----
guided_summary = []
for _, row in guided_info.iterrows():
    t = row["Topic"]
    if t == -1:
        guided_summary.append({"Topic": -1, "Seed Category": "(Outlier / Unclassified)",
                               "Count": row["Count"], "Discovered Keywords": ""})
    else:
        kws = [w for w, _ in guided_model.get_topic(t)[:8]]
        seed_cat = seed_labels[t] if t < len(seed_labels) else "✨ Emergent"
        guided_summary.append({
            "Topic": t,
            "Seed Category": seed_cat,
            "Count": row["Count"],
            "Discovered Keywords": ", ".join(kws),
        })

guided_df = pd.DataFrame(guided_summary)
print("\n" + "="*90)
print("GUIDED TOPIC SUMMARY  (seeded from Root_Cause_Category)")
print("Seeded topics (0…N) align to seed list in order; higher topics are emergent.")
print("="*90)
display(guided_df)

# ---- 3D scatter coloured by guided topic ----
umap_3d = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
               metric="cosine", random_state=42)
coords = umap_3d.fit_transform(embs)

topic_names = {}
for _, row in guided_df.iterrows():
    t = int(row["Topic"])
    topic_names[t] = "Outlier" if t == -1 else f"{t}: {row['Seed Category']}"

plot_df = pd.DataFrame({
    "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
    "topic_name": [topic_names.get(t, f"Topic {t}") for t in guided_topics],
    "pr_id": pdf["pr_id"].values,
    "event_description": [d[:250] + ("..." if len(d) > 250 else "") for d in docs],
})

fig = px.scatter_3d(
    plot_df, x="x", y="y", z="z", color="topic_name",
    hover_data={"pr_id": True, "event_description": True, "x": False, "y": False, "z": False},
    title=f"Guided BERTopic — {n_guided} topics ({min(n_guided, len(seed_labels))} seeded + {max(0, n_guided-len(seed_labels))} emergent)",
    opacity=0.75, height=700,
)
fig.update_traces(marker_size=3.5)
fig.update_layout(
    legend_title_text="Topic (Seed Category)",
    scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""),
)
fig.show()

In [0]:
# ============================================================================
# Zero-Shot BERTopic — Predefined deviation category labels
#
# Instead of discovering topics unsupervised, provide candidate labels
# explicitly. BERTopic embeds each label, computes cosine similarity to each
# document embedding, and assigns docs above threshold to the matching label.
# Remaining docs (below threshold) go through normal UMAP + HDBSCAN clustering.
#
# Uses GTE-Large-en v1.5 to embed BOTH labels and docs in the same space.
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
from pyspark.sql import functions as F
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# ---- Candidate topic labels (domain-informed, based on Root_Cause_Category + known deviation types) ----
candidate_topics = [
    "Inadvertent unblinding of clinical trial treatment assignment",
    "GMP manufacturing deviation or batch failure",
    "Labeling and packaging error",
    "Audit finding or inspection observation",
    "Stability testing out of specification result",
    "Data integrity or GCP compliance issue",
    "Training non-compliance or qualification gap",
    "CAPA corrective and preventive action overdue",
    "Clinical protocol deviation or non-compliance",
    "Vendor or supplier quality issue",
    "Document control or SOP deviation",
    "Equipment or instrument malfunction or calibration",
    "Environmental monitoring excursion",
    "Product complaint or adverse event reporting",
    "IT system validation or computerized system issue",
    "Material or component quality failure",
    "Cleaning or contamination control deviation",
    "Transportation or cold chain deviation",
]
print(f"Candidate topics defined: {len(candidate_topics)}")
for i, t in enumerate(candidate_topics):
    print(f"  {i:>2}. {t}")

# ---- Load a lightweight CPU-only model for BOTH labels and docs ----
# all-MiniLM-L6-v2: 384-dim, max 256 tokens, no custom code, CPU-safe.
# Re-embeds docs in the same space as labels so cosine similarity is valid.
# (The precomputed GTE/BGE-M3 embeddings can't be used here because zero-shot
# requires labels and docs in the SAME embedding space.)
zs_embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print(f"\nModel loaded (CPU): all-MiniLM-L6-v2  |  dim={zs_embed_model.get_sentence_embedding_dimension()}")

# ---- Prepare docs and compute fresh embeddings on CPU ----
zs_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "embedding_text")
    .toPandas()
)
zs_pdf["embedding_text"] = zs_pdf["embedding_text"].fillna("")
zs_docs = [_strip_for_tfidf(t) for t in zs_pdf["embedding_text"].tolist()]

import time
t0 = time.time()
zs_embs = zs_embed_model.encode(zs_docs, batch_size=64, show_progress_bar=True,
                                 normalize_embeddings=True)
print(f"Encoded {len(zs_docs):,} docs in {time.time()-t0:.1f}s  |  shape: {zs_embs.shape}")

# ---- Fit zero-shot BERTopic ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

IGNORE_WORDS = {
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text", "other",
    "not", "applicable", "the", "and", "for", "was", "were", "that", "this",
    "with", "from", "but", "are", "has", "have", "been", "will",
}
custom_stops = list(ENGLISH_STOP_WORDS) + list(IGNORE_WORDS)

zs_model = BERTopic(
    embedding_model=zs_embed_model,             # embeds candidate labels in same MiniLM space
    umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    zeroshot_topic_list=candidate_topics,      # ← ZERO-SHOT: pre-defined labels
    zeroshot_min_similarity=0.5,               # cosine sim threshold for assignment
    calculate_probabilities=True,
    verbose=True,
)

zs_topics, zs_probs = zs_model.fit_transform(zs_docs, embeddings=zs_embs)

# ---- Results ----
zs_info = zs_model.get_topic_info()
n_zs = len(zs_info[zs_info.Topic != -1])
n_out = int((np.array(zs_topics) == -1).sum())
n_assigned = len(zs_topics) - n_out
print(f"\nZero-shot results:")
print(f"  Topics found: {n_zs}  |  Assigned: {n_assigned} ({100*n_assigned/len(zs_topics):.1f}%)  |  Outliers: {n_out} ({100*n_out/len(zs_topics):.1f}%)")

# ---- Summary table ----
zs_summary = []
for _, row in zs_info.iterrows():
    t = row["Topic"]
    if t == -1:
        zs_summary.append({"Topic": -1, "Label": "(Outlier / Unclassified)",
                           "Count": row["Count"], "Keywords": "", "Source": ""})
    else:
        kws = [w for w, _ in zs_model.get_topic(t)[:8]]
        label = row.get("Name", "") or f"Topic {t}"
        # Determine if this topic came from zero-shot or clustering
        source = "Zero-shot" if t < len(candidate_topics) else "✨ Emergent (clustered)"
        zs_summary.append({
            "Topic": t, "Label": label,
            "Count": row["Count"],
            "Keywords": ", ".join(kws),
            "Source": source,
        })

zs_df = pd.DataFrame(zs_summary)
print("\n" + "="*95)
print("ZERO-SHOT TOPIC SUMMARY")
print("Docs assigned by cosine similarity to candidate labels (threshold=0.5).")
print("Remaining docs clustered via UMAP + HDBSCAN as emergent topics.")
print("="*95)
display(zs_df)

# ---- 3D scatter ----
umap_3d = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
               metric="cosine", random_state=42)
coords = umap_3d.fit_transform(zs_embs)

topic_labels_map = {}
for _, row in zs_df.iterrows():
    t = int(row["Topic"])
    lbl = row["Label"]
    topic_labels_map[t] = "Outlier" if t == -1 else f"{t}: {lbl[:45]}"

plot_df = pd.DataFrame({
    "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
    "topic_label": [topic_labels_map.get(t, f"Topic {t}") for t in zs_topics],
    "pr_id": zs_pdf["pr_id"].values,
    "event_description": [d[:250] + ("..." if len(d) > 250 else "") for d in zs_docs],
})

fig = px.scatter_3d(
    plot_df, x="x", y="y", z="z", color="topic_label",
    hover_data={"pr_id": True, "event_description": True, "x": False, "y": False, "z": False},
    title=f"Zero-Shot BERTopic — {n_zs} topics ({len(candidate_topics)} candidates, threshold=0.5)",
    opacity=0.75, height=700,
)
fig.update_traces(marker_size=3.5)
fig.update_layout(
    legend_title_text="Topic",
    scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""),
)
fig.show()

In [0]:
# ============================================================================
# Multi-Label Topic Assignment via approximate_distribution()
#
# BERTopic's approximate_distribution uses a sliding window over each document
# to compute a soft probability distribution across ALL topics. This means one
# event can have meaningful probability in multiple topics simultaneously.
#
# Uses the fitted zs_model from the previous cell.
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
import plotly.figure_factory as ff

# ---- Compute approximate topic distributions (sliding window) ----
print("Computing approximate distributions (sliding window over documents)...")
topic_distr, _ = zs_model.approximate_distribution(zs_docs, min_similarity=0.01)
print(f"Distribution matrix shape: {topic_distr.shape}  (docs × topics)")

# ---- Multi-label assignment: for each doc, find ALL topics above threshold ----
THRESHOLD = 0.1  # topic must have >= 10% probability to be considered "present"

# Build topic label lookup (excluding outlier topic -1)
topic_id_to_label = {}
for _, row in zs_df.iterrows():
    t = int(row["Topic"])
    if t >= 0:
        # Use a short label
        lbl = row["Label"]
        # Strip BERTopic's auto-prefix like "0_keyword_keyword_keyword"
        if lbl.startswith(f"{t}_"):
            kws = lbl.split("_")[1:4]
            lbl = " / ".join(kws)
        topic_id_to_label[t] = lbl

# Get column indices (approximate_distribution columns = topics in order, excluding -1)
topic_ids = sorted(topic_id_to_label.keys())

# For each doc, find topics above threshold
multi_labels = []
for i in range(len(zs_docs)):
    doc_probs = topic_distr[i]
    assigned = []
    for col_idx, prob in enumerate(doc_probs):
        if prob >= THRESHOLD and col_idx < len(topic_ids):
            assigned.append((topic_ids[col_idx], prob))
    # Sort by probability descending
    assigned.sort(key=lambda x: x[1], reverse=True)
    multi_labels.append(assigned)

# ---- Statistics ----
n_topics_per_doc = [len(ml) for ml in multi_labels]
print(f"\n{'='*70}")
print(f"MULTI-LABEL STATISTICS  (threshold = {THRESHOLD})")
print(f"{'='*70}")
print(f"  Docs with 0 topics (below threshold everywhere): {n_topics_per_doc.count(0)}")
print(f"  Docs with exactly 1 topic:                       {n_topics_per_doc.count(1)}")
print(f"  Docs with 2 topics:                              {n_topics_per_doc.count(2)}")
print(f"  Docs with 3 topics:                              {n_topics_per_doc.count(3)}")
print(f"  Docs with 4+ topics:                             {sum(1 for x in n_topics_per_doc if x >= 4)}")
print(f"  Average topics per doc:                          {np.mean(n_topics_per_doc):.2f}")
print(f"  Max topics on a single doc:                      {max(n_topics_per_doc)}")

# ---- Show example multi-topic events ----
print(f"\n{'='*70}")
print("EXAMPLE EVENTS SPANNING MULTIPLE TOPICS")
print(f"{'='*70}")

multi_topic_indices = [(i, ml) for i, ml in enumerate(multi_labels) if len(ml) >= 2]
multi_topic_indices.sort(key=lambda x: len(x[1]), reverse=True)

for idx, (i, topics_assigned) in enumerate(multi_topic_indices[:15]):
    pr_id = zs_pdf.iloc[i]["pr_id"]
    print(f"\n  [{idx+1}] pr_id={pr_id}  ({len(topics_assigned)} topics)")
    for t_id, prob in topics_assigned:
        print(f"       {prob:.2f}  →  {topic_id_to_label.get(t_id, f'Topic {t_id}')}")
    # Show first 150 chars of text
    print(f"       Text: {zs_docs[i][:150]}...")

# ---- Topic co-occurrence matrix ----
print(f"\n{'='*70}")
print("TOPIC CO-OCCURRENCE HEATMAP")
print(f"{'='*70}")

n_topics_total = len(topic_ids)
cooccurrence = np.zeros((n_topics_total, n_topics_total), dtype=int)

for ml in multi_labels:
    if len(ml) >= 2:
        present_topics = [t_id for t_id, _ in ml]
        for a in range(len(present_topics)):
            for b in range(a + 1, len(present_topics)):
                idx_a = topic_ids.index(present_topics[a])
                idx_b = topic_ids.index(present_topics[b])
                cooccurrence[idx_a][idx_b] += 1
                cooccurrence[idx_b][idx_a] += 1

# Filter to topics that actually co-occur with something
active_mask = cooccurrence.sum(axis=0) > 0
active_indices = np.where(active_mask)[0]

if len(active_indices) > 0:
    co_sub = cooccurrence[np.ix_(active_indices, active_indices)]
    labels_sub = [topic_id_to_label.get(topic_ids[i], f"T{topic_ids[i]}")[:30] for i in active_indices]

    fig = px.imshow(
        co_sub,
        x=labels_sub, y=labels_sub,
        color_continuous_scale="Blues",
        title=f"Topic Co-occurrence (events spanning 2+ topics, threshold={THRESHOLD})",
        labels={"color": "Count"},
        height=700, width=800,
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
else:
    print("  No co-occurrences found at this threshold.")

# ---- Summary DataFrame for export ----
multi_label_records = []
for i, ml in enumerate(multi_labels):
    if len(ml) >= 1:
        multi_label_records.append({
            "pr_id": zs_pdf.iloc[i]["pr_id"],
            "n_topics": len(ml),
            "primary_topic": topic_id_to_label.get(ml[0][0], f"Topic {ml[0][0]}") if ml else None,
            "primary_prob": ml[0][1] if ml else 0,
            "all_topics": " | ".join(f"{topic_id_to_label.get(t, f'T{t}')} ({p:.2f})" for t, p in ml),
        })

multi_label_df = pd.DataFrame(multi_label_records)
print(f"\nMulti-label DataFrame: {len(multi_label_df)} docs with at least 1 topic assigned")
display(multi_label_df.sort_values("n_topics", ascending=False).head(20))

In [0]:
# ============================================================================
# LLM-Based Topic Assignment
#
# Instead of clustering + c-TF-IDF keywords, directly ask an LLM to read each
# deviation event's embedding_text and assign one or more topic labels.
# Uses Databricks Foundation Model API (Meta Llama 3.1 70B Instruct).
#
# Multi-label by design: the LLM returns a primary topic + optional secondary.
# ============================================================================
import json, time, os
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ---- Setup client ----
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints",
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"

# ---- Topic taxonomy ----
TOPIC_LIST = [
    "Inadvertent unblinding",
    "GMP manufacturing deviation",
    "Labeling and packaging error",
    "Audit finding or inspection observation",
    "Stability testing OOS",
    "Data integrity or GCP issue",
    "Training non-compliance",
    "CAPA overdue or inadequate",
    "Clinical protocol deviation",
    "Vendor or supplier quality issue",
    "Document control or SOP deviation",
    "Equipment or instrument malfunction",
    "Environmental monitoring excursion",
    "Product complaint or adverse event",
    "IT system or CSV issue",
    "Material or component failure",
    "Cleaning or contamination control",
    "Transportation or cold chain deviation",
    "Pharmacovigilance or safety reporting",
    "Regulatory submission or labeling update",
    "Clinical study operations",
]

TOPIC_LIST_STR = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(TOPIC_LIST))

SYSTEM_PROMPT = """You are a pharmaceutical quality deviation classifier at Takeda. 
Given a deviation event description, assign the most relevant topic(s) from the provided list.
Return ONLY valid JSON with no extra text."""

USER_TEMPLATE = """Classify this deviation event. Assign 1-3 topics from the list below.

Topics:
{topics}

Return JSON:
{{"primary": "<exact topic name from list>", "secondary": ["<topic>", ...], "reasoning": "<1 sentence>"}}

Deviation text:
{text}"""

# ---- Robust JSON extraction + classify with backoff ----
import re

def _extract_json(text):
    """Robustly extract JSON from LLM response (handles markdown, extra text)."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Strip markdown code fences
    m = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Find first { ... } block
    m = re.search(r'(\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\})', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    raise ValueError(f"No valid JSON found in: {text[:150]}")

def classify_doc(pr_id, text, max_retries=4):
    """Call LLM to classify a single deviation event with exponential backoff."""
    truncated = text[:3000] if len(text) > 3000 else text
    user_msg = USER_TEMPLATE.format(topics=TOPIC_LIST_STR, text=truncated)
    
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=250,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            parsed = _extract_json(raw)
            return {
                "pr_id": pr_id,
                "primary": parsed.get("primary", "Unknown"),
                "secondary": parsed.get("secondary", []),
                "reasoning": parsed.get("reasoning", ""),
                "raw": raw,
                "error": None,
            }
        except Exception as e:
            if attempt == max_retries:
                return {
                    "pr_id": pr_id,
                    "primary": "ERROR",
                    "secondary": [],
                    "reasoning": "",
                    "raw": str(e)[:300],
                    "error": str(e)[:300],
                }
            time.sleep(2 ** attempt)  # 1s, 2s, 4s, 8s backoff

# ---- Load docs ----
llm_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "embedding_text")
    .toPandas()
)
llm_pdf["embedding_text"] = llm_pdf["embedding_text"].fillna("")
print(f"Documents to classify: {len(llm_pdf):,}")
print(f"Model: {MODEL}")
print(f"Topics: {len(TOPIC_LIST)}")

# ---- Run classification (lower concurrency to respect rate limits) ----
MAX_WORKERS = 4
results = []

t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(classify_doc, row["pr_id"], row["embedding_text"]): i
        for i, row in llm_pdf.iterrows()
    }
    
    done_count = 0
    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        done_count += 1
        if done_count % 100 == 0:
            elapsed = time.time() - t0
            rate = done_count / elapsed
            eta = (len(llm_pdf) - done_count) / rate
            print(f"  {done_count:>5}/{len(llm_pdf)}  ({rate:.1f} docs/s, ETA {eta:.0f}s)")

elapsed = time.time() - t0
print(f"\nClassification complete: {len(results):,} docs in {elapsed:.1f}s ({len(results)/elapsed:.1f} docs/s)")

# ---- Build results DataFrame ----
llm_results_df = pd.DataFrame(results)
n_errors = llm_results_df["error"].notna().sum()
print(f"Errors: {n_errors} ({100*n_errors/len(llm_results_df):.1f}%)")

# ---- Topic distribution ----
print(f"\n{'='*80}")
print("LLM TOPIC ASSIGNMENT SUMMARY")
print(f"{'='*80}")

primary_counts = llm_results_df["primary"].value_counts()
print(f"\nPrimary topic distribution ({len(primary_counts)} unique):")
for topic, count in primary_counts.head(25).items():
    print(f"  {count:>4}  {topic}")

# Multi-label stats
llm_results_df["n_topics"] = llm_results_df.apply(
    lambda r: 1 + len(r["secondary"]) if r["primary"] != "ERROR" else 0, axis=1
)
print(f"\nMulti-label stats:")
print(f"  1 topic only: {(llm_results_df['n_topics'] == 1).sum()}")
print(f"  2 topics:     {(llm_results_df['n_topics'] == 2).sum()}")
print(f"  3 topics:     {(llm_results_df['n_topics'] == 3).sum()}")
print(f"  4+ topics:    {(llm_results_df['n_topics'] >= 4).sum()}")

# ---- Show examples ----
print(f"\n{'='*80}")
print("EXAMPLE ASSIGNMENTS (first 10 multi-topic events)")
print(f"{'='*80}")
multi = llm_results_df[llm_results_df["n_topics"] >= 2].head(10)
for _, row in multi.iterrows():
    print(f"\n  pr_id={row['pr_id']}")
    print(f"    Primary:   {row['primary']}")
    print(f"    Secondary: {row['secondary']}")
    print(f"    Reasoning: {row['reasoning']}")

# ---- Display full table ----
display(llm_results_df[["pr_id", "primary", "secondary", "n_topics", "reasoning"]]
        .sort_values("n_topics", ascending=False).head(30))

In [0]:
# ============================================================================
# LLM-Based Topic Assignment (Hybrid: fixed taxonomy + novel-topic discovery)
#
# Per-document, multi-label classification.
#   - Prefers the fixed TOPIC_LIST when a listed topic fits.
#   - Lets the LLM INVENT a new concise topic when nothing fits.
#   - Adds controlled root-cause & impact vocabularies, evidence span,
#     confidence rubric, grounding guardrails.
#   - Normalizes/snaps returned labels to canonical spellings; tracks novel ones.
#
# Databricks Foundation Model API (Meta Llama 3.3 70B Instruct).
# ============================================================================
import json, time, os, re
import numpy as np, pandas as pd
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ---- Setup client ----
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(api_key=token, base_url=f"{host}/serving-endpoints")
MODEL = "databricks-meta-llama-3-3-70b-instruct"

# ---- Controlled vocabularies ----
TOPIC_LIST = [
    "Inadvertent unblinding",
    "GMP manufacturing deviation",
    "Labeling and packaging error",
    "Audit finding or inspection observation",
    "Stability testing OOS",
    "Data integrity or GCP issue",
    "Training non-compliance",
    "CAPA overdue or inadequate",
    "Clinical protocol deviation",
    "Vendor or supplier quality issue",
    "Document control or SOP deviation",
    "Equipment or instrument malfunction",
    "Environmental monitoring excursion",
    "Product complaint or adverse event",
    "IT system or CSV issue",
    "Material or component failure",
    "Cleaning or contamination control",
    "Transportation or cold chain deviation",
    "Pharmacovigilance or safety reporting",
    "Regulatory submission or labeling update",
    "Clinical study operations",
]

ROOT_CAUSE_LIST = [
    "Human Error", "Training", "Process/Procedure", "Vendor/CRO",
    "System/Technology (IRT/eCRF)", "Documentation", "Clinical Operations",
    "Laboratory", "Manufacturing/Supply", "Not determinable", "Other",
]

IMPACT_LIST = [
    "Patient Safety", "Data Integrity", "Trial Blinding/Unblinding",
    "GCP Compliance", "Product Quality", "Supply Chain",
    "Documentation/Records", "Not determinable", "None",
]

TOPIC_LIST_STR = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(TOPIC_LIST))
ROOT_CAUSE_STR = " | ".join(ROOT_CAUSE_LIST)
IMPACT_STR     = " | ".join(IMPACT_LIST)

# ---- Prompts ----
SYSTEM_PROMPT = """You are a GxP pharmaceutical quality deviation classifier at Takeda,
specializing in clinical trial deviations and QMS records.
Prefer the provided topic list, but you MAY create a new concise topic when no
listed topic fits. Work ONLY from evidence in the text; if information is not
stated, use "Not determinable" rather than guessing. Do not invent facts,
systems, or root causes. Return ONLY valid JSON, no extra text."""

USER_TEMPLATE = """Classify this deviation event.

STEP 1 - Choose a PRIMARY topic:
  - First try to match one topic from the TOPICS list below.
  - ONLY if no listed topic is a good fit, CREATE a new concise topic
    label (2-5 words) that captures the underlying business process.
STEP 2 - Assign 0-2 SECONDARY topics (listed or newly created, same rule).
STEP 3 - root_cause_category from ROOT CAUSE list.
STEP 4 - regulatory_impact from IMPACT list.
STEP 5 - Quote a short evidence_span justifying the primary topic.
STEP 6 - Score confidence per RUBRIC.
STEP 7 - List any topics you CREATED (not from the list) in "novel_topics".

Rules:
- Prefer listed topics; only invent when clearly necessary.
- Newly created labels must be general business themes, NOT product/study names.
- Keep invented labels concise and reusable.

TOPICS:
{topics}

ROOT CAUSE: {root_causes}

IMPACT: {impacts}

CONFIDENCE RUBRIC:
90-100 = explicit statement in text
70-89  = strong implication
40-69  = partial/indirect evidence
<40    = weak or inferred; text is sparse

Return JSON exactly:
{{"primary": "<listed or new topic>",
  "secondary": ["<topic>", ...],
  "novel_topics": ["<any topic you created>", ...],
  "root_cause_category": "<from ROOT CAUSE>",
  "regulatory_impact": "<from IMPACT>",
  "evidence_span": "<short quote>",
  "confidence": <integer 0-100>,
  "reasoning": "<1 sentence>"}}

Deviation text:
{text}"""

# ---- Canonical lookup + normalizer ----
_TOPIC_CANON = {t.lower(): t for t in TOPIC_LIST}

def _normalize_topic(label):
    """Snap to canonical list if known; else mark as novel. Returns (label, is_known)."""
    if not label or not isinstance(label, str):
        return None, False
    key = label.strip().lower()
    if key in _TOPIC_CANON:
        return _TOPIC_CANON[key], True      # known -> canonical spelling
    return label.strip(), False             # novel -> keep model's wording

# ---- Robust JSON extraction ----
def _extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    m = re.search(r'(\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\})', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    raise ValueError(f"No valid JSON found in: {text[:150]}")

# ---- Classify one doc with exponential backoff ----
def classify_doc(pr_id, text, max_retries=4):
    truncated = text[:3000] if len(text) > 3000 else text
    user_msg = USER_TEMPLATE.format(
        topics=TOPIC_LIST_STR, root_causes=ROOT_CAUSE_STR,
        impacts=IMPACT_STR, text=truncated,
    )
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=350,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            parsed = _extract_json(raw)

            # Normalize topics: snap known ones, flag novel ones
            prim_label, prim_known = _normalize_topic(parsed.get("primary", "Unknown"))
            sec_norm = [_normalize_topic(s) for s in (parsed.get("secondary", []) or [])]
            secondary = [lbl for lbl, _ in sec_norm if lbl]

            novel = []
            if prim_label and not prim_known:
                novel.append(prim_label)
            novel += [lbl for lbl, known in sec_norm if lbl and not known]
            # merge with model's self-reported novel_topics, de-duped
            novel = sorted(set(novel) | set(parsed.get("novel_topics", []) or []))

            return {
                "pr_id": pr_id,
                "primary": prim_label,
                "primary_is_novel": (prim_label is not None) and (not prim_known),
                "secondary": secondary,
                "novel_topics": novel,
                "root_cause_category": parsed.get("root_cause_category", "Not determinable"),
                "regulatory_impact": parsed.get("regulatory_impact", "Not determinable"),
                "evidence_span": parsed.get("evidence_span", ""),
                "confidence": parsed.get("confidence", None),
                "reasoning": parsed.get("reasoning", ""),
                "raw": raw,
                "error": None,
            }
        except Exception as e:
            if attempt == max_retries:
                return {
                    "pr_id": pr_id, "primary": "ERROR", "primary_is_novel": False,
                    "secondary": [], "novel_topics": [],
                    "root_cause_category": "ERROR", "regulatory_impact": "ERROR",
                    "evidence_span": "", "confidence": None, "reasoning": "",
                    "raw": str(e)[:300], "error": str(e)[:300],
                }
            time.sleep(2 ** attempt)  # 1s, 2s, 4s, 8s

# ---- Load docs ----
llm_pdf = spark.table(EMB_TABLE).select("pr_id", "embedding_text").toPandas()
llm_pdf["embedding_text"] = llm_pdf["embedding_text"].fillna("")
print(f"Documents to classify: {len(llm_pdf):,} | Model: {MODEL} | Topics: {len(TOPIC_LIST)}")

# ---- Run classification (concurrency tuned to respect rate limits) ----
MAX_WORKERS = 4
results, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(classify_doc, r["pr_id"], r["embedding_text"]): i
               for i, r in llm_pdf.iterrows()}
    done = 0
    for future in as_completed(futures):
        results.append(future.result())
        done += 1
        if done % 100 == 0:
            el = time.time() - t0; rate = done / el
            print(f"  {done:>5}/{len(llm_pdf)}  ({rate:.1f} docs/s, ETA {(len(llm_pdf)-done)/rate:.0f}s)")
el = time.time() - t0
print(f"\nDone: {len(results):,} docs in {el:.1f}s ({len(results)/max(el,1e-9):.1f} docs/s)")

# ---- Results DataFrame ----
llm_results_df = pd.DataFrame(results)
n_err = llm_results_df["error"].notna().sum()
print(f"Errors: {n_err} ({100*n_err/len(llm_results_df):.1f}%)")

# ---- Multi-label stats ----
llm_results_df["n_topics"] = llm_results_df.apply(
    lambda r: 1 + len(r["secondary"]) if r["primary"] != "ERROR" else 0, axis=1)

# ---- Rollups for dashboard ----
print("\n" + "="*80)
print("PRIMARY TOPIC DISTRIBUTION")
print("="*80)
print(llm_results_df["primary"].value_counts().head(30).to_string())

print("\n" + "="*80)
print("ROOT CAUSE DISTRIBUTION")
print("="*80)
print(llm_results_df["root_cause_category"].value_counts().to_string())

print("\n" + "="*80)
print("REGULATORY IMPACT DISTRIBUTION")
print("="*80)
print(llm_results_df["regulatory_impact"].value_counts().to_string())

# ---- Confidence health check ----
conf = pd.to_numeric(llm_results_df["confidence"], errors="coerce")
print("\n" + "="*80)
print("CONFIDENCE")
print("="*80)
print(f"  mean={conf.mean():.1f}  median={conf.median():.1f}  <40 (needs review): {(conf<40).sum()}")

# ---- Multi-label summary ----
print("\n" + "="*80)
print("MULTI-LABEL STATS")
print("="*80)
for k in [1, 2, 3]:
    print(f"  {k} topic(s): {(llm_results_df['n_topics']==k).sum()}")
print(f"  4+ topics:  {(llm_results_df['n_topics']>=4).sum()}")

# ---- Novel (LLM-invented) topics: taxonomy-expansion candidates ----
novel_counter = Counter()
for _, r in llm_results_df.iterrows():
    novel_counter.update(r["novel_topics"] or [])

print("\n" + "="*80)
print("NOVEL TOPICS (not in TOPIC_LIST)")
print("="*80)
print(f"Rows using a novel PRIMARY topic: {llm_results_df['primary_is_novel'].sum()}")
if novel_counter:
    for topic, cnt in novel_counter.most_common(30):
        print(f"  {cnt:>4}  {topic}")
else:
    print("  (none — every record matched the existing taxonomy)")

# ---- Taxonomy coverage ----
valid = (llm_results_df["primary"] != "ERROR").sum()
known_primary = (~llm_results_df["primary_is_novel"] &
                 (llm_results_df["primary"] != "ERROR")).sum()
print(f"\nTaxonomy coverage (primary): {known_primary}/{valid} "
      f"({100*known_primary/max(valid,1):.1f}%)")

# ---- Display: low-confidence first for QA ----
display(
    llm_results_df[["pr_id", "primary", "primary_is_novel", "secondary",
                    "novel_topics", "root_cause_category", "regulatory_impact",
                    "confidence", "n_topics", "evidence_span", "reasoning"]]
    .sort_values("confidence", ascending=True, na_position="first")
    .head(30)
)

In [0]:
# Quick check: what were the errors from the previous run?
errors = llm_results_df[llm_results_df["error"].notna()]
print(f"Total errors: {len(errors)} / {len(llm_results_df)}")
print(f"\nFirst 5 unique error messages:")
for i, msg in enumerate(errors["raw"].unique()[:5]):
    print(f"\n--- Error {i+1} ---")
    print(msg[:500])

In [0]:
# ============================================================================
# GTE-Large-en v1.5 vs BGE-M3 — Topic Modeling Comparison
# Both models produce 1024-dim embeddings on core_embedding_text.
# Same BERTopic config (best UMAP params, same HDBSCAN, same c-TF-IDF).
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---- Load both embedding columns from the same table ----
comp_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "embedding_text", "core_embedding", "core_embedding_s2v")
    .toPandas()
)
comp_pdf["embedding_text"] = comp_pdf["embedding_text"].fillna("")

# Clean docs for c-TF-IDF (reuse _strip_for_tfidf from Cell 2)
comp_docs = [_strip_for_tfidf(t) for t in comp_pdf["embedding_text"].tolist()]

embs_bge_core = np.array(comp_pdf["core_embedding"].tolist(), dtype=np.float32)
embs_gte      = np.array(comp_pdf["core_embedding_s2v"].tolist(), dtype=np.float32)

print(f"Loaded {len(comp_docs):,} docs")
print(f"  BGE-M3 core_embedding:          {embs_bge_core.shape}")
print(f"  GTE-Large-en core_embedding_s2v: {embs_gte.shape}")

# ---- Shared config (best UMAP params from grid search) ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number",
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS

def _fit_bertopic(embeddings, label):
    """Fit BERTopic on given embeddings and return (topics, model, metrics)."""
    model = BERTopic(
        embedding_model=None,
        umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                        metric="cosine", random_state=42),
        hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                              metric="euclidean", cluster_selection_method="eom",
                              prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
        calculate_probabilities=True, verbose=False,
    )
    topics_i, probs_i = model.fit_transform(comp_docs, embeddings=embeddings)

    n_topics = len(set(topics_i)) - (1 if -1 in topics_i else 0)
    outlier_pct = 100 * (np.array(topics_i) == -1).mean()
    avg_prob = probs_i.max(axis=1).mean() if probs_i is not None else 0.0
    all_words = []
    for t in set(topics_i):
        if t == -1: continue
        all_words.extend([w for w, _ in model.get_topic(t)[:5]])
    diversity = len(set(all_words)) / max(len(all_words), 1)

    return topics_i, model, {
        "Model": label, "Topics": n_topics,
        "Outlier %": round(outlier_pct, 1),
        "Avg Prob": round(avg_prob, 4),
        "Diversity": round(diversity, 4),
    }

print("\nFitting BGE-M3 (core tier)...")
topics_bge, model_bge, m_bge = _fit_bertopic(embs_bge_core, "BGE-M3 (core)")

print("Fitting GTE-Large-en v1.5...")
topics_gte, model_gte, m_gte = _fit_bertopic(embs_gte, "GTE-Large-en v1.5")

# ---- Comparison metrics table ----
comp_table = pd.DataFrame([m_bge, m_gte])
print("\n" + "="*65)
print("MODEL COMPARISON  (same UMAP/HDBSCAN params, same c-TF-IDF text)")
print("="*65)
display(comp_table)

# ---- Clustering agreement ----
ari = adjusted_rand_score(topics_bge, topics_gte)
nmi = normalized_mutual_info_score(topics_bge, topics_gte)
print(f"\nClustering Agreement (BGE-M3 vs GTE):")
print(f"  Adjusted Rand Index:           {ari:.4f}  (1.0 = identical, 0.0 = random)")
print(f"  Normalized Mutual Information:  {nmi:.4f}  (1.0 = identical partitions)")

# ---- Side-by-side 3D UMAP scatter ----
umap_3d_viz = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
                   metric="cosine", random_state=42)

coords_bge = umap_3d_viz.fit_transform(embs_bge_core)
coords_gte = umap_3d_viz.fit_transform(embs_gte)

desc_preview = [d[:250] + ("..." if len(d) > 250 else "") for d in comp_docs]

# BGE-M3 plot
plot_bge = pd.DataFrame({
    "UMAP-1": coords_bge[:, 0], "UMAP-2": coords_bge[:, 1], "UMAP-3": coords_bge[:, 2],
    "topic": [str(t) for t in topics_bge],
    "pr_id": comp_pdf["pr_id"].values,
    "event_description": desc_preview,
})
fig1 = px.scatter_3d(
    plot_bge, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="topic",
    hover_data={"pr_id": True, "event_description": True, "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"BGE-M3 (core) — {m_bge['Topics']} topics, {m_bge['Outlier %']}% outlier",
    opacity=0.7, height=650,
)
fig1.update_traces(marker_size=3)
fig1.show()

# GTE plot
plot_gte = pd.DataFrame({
    "UMAP-1": coords_gte[:, 0], "UMAP-2": coords_gte[:, 1], "UMAP-3": coords_gte[:, 2],
    "topic": [str(t) for t in topics_gte],
    "pr_id": comp_pdf["pr_id"].values,
    "event_description": desc_preview,
})
fig2 = px.scatter_3d(
    plot_gte, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="topic",
    hover_data={"pr_id": True, "event_description": True, "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"GTE-Large-en v1.5 — {m_gte['Topics']} topics, {m_gte['Outlier %']}% outlier",
    opacity=0.7, height=650,
)
fig2.update_traces(marker_size=3)
fig2.show()

# ---- Top-5 keywords comparison for shared top topics ----
print("\nTop 5 keywords per model (first 10 topics):")
print(f"{'Topic':<8} {'BGE-M3 (core)':<55} {'GTE-Large-en v1.5'}")
print("-" * 120)
for t in range(min(10, m_bge["Topics"], m_gte["Topics"])):
    kw_bge = ", ".join(w for w, _ in model_bge.get_topic(t)[:5])
    kw_gte = ", ".join(w for w, _ in model_gte.get_topic(t)[:5])
    print(f"{t:<8} {kw_bge:<55} {kw_gte}")

In [0]:
# ============================================================================
# Cell 3 — Inspect topics + attach labels back to pr_id
# ============================================================================
# Top keywords per topic
for t in sorted(set(topics)):
    if t == -1:
        continue
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
    print(f"Topic {t:>2}: {words}")

# Map topic + representative keywords back onto each deviation
pdf["topic"] = topics
pdf["topic_prob"] = probs.max(axis=1) if probs is not None else np.nan

# Write back to Delta so the dashboard (D6) can join on pr_id
out_df = spark.createDataFrame(
    pdf[["pr_id", "topic", "topic_prob"]].astype({"pr_id": str, "topic": int})
)
(out_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{ALYT}.deviation_topics"))
print(f"✓ wrote {CATALOG}.{ALYT}.deviation_topics")